### RAG Pipelines - Data Ingestion to vector DB Pipeline

In [1]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\chath\AppData\Local\Temp\ipykernel_20244\1797134647.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: airtel.pdf
  ✓ Loaded 1 pages

Processing: hostel.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'iText® Core 8.0.3 (AGPL version) ©2000-2024 Apryse Group NV', 'creator': 'PyPDF', 'creationdate': '2026-08-30T19:18:23+05:30', 'moddate': '2026-08-30T19:18:23+05:30', 'source': '..\\data\\pdf\\airtel.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'airtel.pdf', 'file_type': 'pdf'}, page_content='Sampath Vishwa Internet Banking - One Time Fund Transfer\nStatus : Success\nAmount : LKR 200.00\nTransaction ID : 83934321\nDate & Time : 30/08/2026 07:18 PM\nSource Account : ********2499\nBeneficiary Account : 8025032570\nBeneficiary Remarks : ccc air\nBeneficiary Bank : COMMERCIAL BANK OF CEYLON\nSampath Bank PLC - Company No. PQ144 \nTel: +94-11-2303050 | Email: info@sampath.lk | Web: www.sampath.lk\nGenerated: 30/08/2026 07:18:23 PM \nThis is a system-generated print. Signature not required.'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-09-03

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

Split 2 documents into 3 chunks

Example chunk:
Content: Sampath Vishwa Internet Banking - One Time Fund Transfer
Status : Success
Amount : LKR 200.00
Transaction ID : 83934321
Date & Time : 30/08/2026 07:18 PM
Source Account : ********2499
Beneficiary Acco...
Metadata: {'producer': 'iText® Core 8.0.3 (AGPL version) ©2000-2024 Apryse Group NV', 'creator': 'PyPDF', 'creationdate': '2026-08-30T19:18:23+05:30', 'moddate': '2026-08-30T19:18:23+05:30', 'source': '..\\data\\pdf\\airtel.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'airtel.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iText® Core 8.0.3 (AGPL version) ©2000-2024 Apryse Group NV', 'creator': 'PyPDF', 'creationdate': '2026-08-30T19:18:23+05:30', 'moddate': '2026-08-30T19:18:23+05:30', 'source': '..\\data\\pdf\\airtel.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'airtel.pdf', 'file_type': 'pdf'}, page_content='Sampath Vishwa Internet Banking - One Time Fund Transfer\nStatus : Success\nAmount : LKR 200.00\nTransaction ID : 83934321\nDate & Time : 30/08/2026 07:18 PM\nSource Account : ********2499\nBeneficiary Account : 8025032570\nBeneficiary Remarks : ccc air\nBeneficiary Bank : COMMERCIAL BANK OF CEYLON\nSampath Bank PLC - Company No. PQ144 \nTel: +94-11-2303050 | Email: info@sampath.lk | Web: www.sampath.lk\nGenerated: 30/08/2026 07:18:23 PM \nThis is a system-generated print. Signature not required.'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-09-03